# 05 三对角方程组与追赶法

三对角矩阵只在主对角线、下对角线和上对角线上有非零元素。自然三次样条、隐式差分和一维边值问题都会产生这类系统。利用结构后，求解复杂度可以从一般稠密矩阵的 $O(n^3)$ 降到 $O(n)$。


In [ ]:
import pathlib
import sys
import time

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["font.sans-serif"] = [
    "Arial Unicode MS",
    "PingFang SC",
    "Heiti SC",
    "SimHei",
    "Noto Sans CJK SC",
]
plt.rcParams["axes.unicode_minus"] = False

ROOT = pathlib.Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "pyproject.toml").exists() and (candidate / "src" / "py_sc").exists():
        ROOT = candidate
        break
SRC = ROOT / "src"
CHAPTER = ROOT / "chapters" / "ch06_direct_linear_systems"
SCRIPTS = CHAPTER / "scripts"
for path in [SRC, SCRIPTS]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from py_sc import thomas_algorithm


## 三对角系统

一般形式为

$$
\begin{bmatrix}
b_1 & c_1 \\
a_2 & b_2 & c_2 \\
& a_3 & b_3 & c_3 \\
& & \ddots & \ddots & \ddots \\
& & & a_n & b_n
\end{bmatrix}
\boldsymbol{x}=\boldsymbol{d}.
$$

消元时只会更新相邻对角线，不会产生远离对角线的新非零元素。


## Thomas 追赶法

追过程定义修正后的上对角和右端项：

$$
c'_1=\frac{c_1}{b_1},\qquad d'_1=\frac{d_1}{b_1},
$$

$$
c'_i=\frac{c_i}{b_i-a_i c'_{i-1}},\qquad
d'_i=\frac{d_i-a_i d'_{i-1}}{b_i-a_i c'_{i-1}}.
$$

赶过程为

$$
x_n=d'_n,\qquad x_i=d'_i-c'_i x_{i+1}.
$$


In [ ]:
def teaching_thomas(lower, diagonal, upper, rhs):
    lower = np.asarray(lower, dtype=float)
    diagonal = np.asarray(diagonal, dtype=float)
    upper = np.asarray(upper, dtype=float)
    rhs = np.asarray(rhs, dtype=float)
    n = diagonal.size
    c_prime = np.zeros(n - 1)
    d_prime = np.zeros(n)
    c_prime[0] = upper[0] / diagonal[0]
    d_prime[0] = rhs[0] / diagonal[0]
    for i in range(1, n):
        pivot = diagonal[i] - lower[i - 1] * c_prime[i - 1]
        if i < n - 1:
            c_prime[i] = upper[i] / pivot
        d_prime[i] = (rhs[i] - lower[i - 1] * d_prime[i - 1]) / pivot
    x = np.zeros(n)
    x[-1] = d_prime[-1]
    for i in range(n - 2, -1, -1):
        x[i] = d_prime[i] - c_prime[i] * x[i + 1]
    return x

n = 6
lower = -np.ones(n - 1)
diagonal = 4.0 * np.ones(n)
upper = -np.ones(n - 1)
A = np.diag(diagonal) + np.diag(lower, -1) + np.diag(upper, 1)
x_exact = np.arange(1.0, n + 1.0)
rhs = A @ x_exact
print("教学版:", teaching_thomas(lower, diagonal, upper, rhs))
print("正式实现:", thomas_algorithm(lower, diagonal, upper, rhs))


## 三次样条系统示例

自然三次样条会产生三对角系统。这里不重新推导第二章的样条公式，只构造一个典型三对角系统，观察追赶法与 dense solve 的结果一致。


In [ ]:
x_nodes = np.linspace(0.0, 1.0, 8)
y_nodes = np.sin(2 * np.pi * x_nodes)
h = np.diff(x_nodes)
n_inner = x_nodes.size - 2
lower = h[1:-1].copy()
diagonal = 2.0 * (h[:-1] + h[1:])
upper = h[1:-1].copy()
rhs = 6.0 * ((y_nodes[2:] - y_nodes[1:-1]) / h[1:] - (y_nodes[1:-1] - y_nodes[:-2]) / h[:-1])
A_spline = np.diag(diagonal) + np.diag(lower, -1) + np.diag(upper, 1)
second_inner = thomas_algorithm(lower, diagonal, upper, rhs)
print("样条内部二阶导数:", second_inner)
print("与 dense solve 差异:", np.linalg.norm(second_inner - np.linalg.solve(A_spline, rhs)))


## 运行时间和复杂度

下面的时间测量只是粗略实验，单次运行会有波动；它用于说明增长趋势，而不是做严格性能评测。


In [ ]:
sizes = [100, 300, 700]
thomas_times = []
dense_times = []
for n in sizes:
    lower = -np.ones(n - 1)
    diagonal = 4.0 * np.ones(n)
    upper = -np.ones(n - 1)
    rhs = np.ones(n)
    A = np.diag(diagonal) + np.diag(lower, -1) + np.diag(upper, 1)
    t0 = time.perf_counter()
    thomas_algorithm(lower, diagonal, upper, rhs)
    thomas_times.append(time.perf_counter() - t0)
    t0 = time.perf_counter()
    np.linalg.solve(A, rhs)
    dense_times.append(time.perf_counter() - t0)

plt.figure(figsize=(7, 4))
plt.loglog(sizes, thomas_times, "o-", label="Thomas 追赶法")
plt.loglog(sizes, dense_times, "s-", label="dense solve")
plt.xlabel("矩阵阶数 n")
plt.ylabel("运行时间（秒，单次粗略测量）")
plt.title("三对角结构带来的效率优势")
plt.grid(True, which="both", alpha=0.3)
plt.legend()
plt.show()

for n, t_thomas, t_dense in zip(sizes, thomas_times, dense_times):
    print(f"n={n:<4d} Thomas={t_thomas:.3e}s dense={t_dense:.3e}s")


## Doolittle、Crout 与待完善内容

三对角 Doolittle 和 Crout 分解都利用同一个事实：带宽在消元过程中保持。Doolittle 通常约定 $L$ 单位对角，Crout 通常约定 $U$ 单位对角。它们的递推变量不同，但本质上都等价于追赶法。

本轮 `scripts` 中保留规范 Thomas 算法实现；三对角 Crout 的独立实现作为后续待完善内容，避免机械复制三份几乎相同的代码。


## 小结

* 三对角结构让消元只作用在相邻对角线上。
* Thomas 追赶法是三对角 LU 分解的紧凑实现。
* 时间和存储复杂度均为 $O(n)$。
* 主元过小仍可能造成风险；对一般带主元三对角问题，需要更谨慎的结构化算法。
